In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from math import log, sqrt, exp
from scipy.stats import norm
import yfinance as yf

In [40]:
#Loading the data
df_27 = pd.read_csv(r'C:\Users\VIDUR\Desktop\PYTHON\My_Project\27-1.csv')
df_28 = pd.read_csv(r'C:\Users\VIDUR\Desktop\PYTHON\My_Project\28-1.csv')
df_29 = pd.read_csv(r'C:\Users\VIDUR\Desktop\PYTHON\My_Project\29-1.csv')

In [41]:
# Data manipulation 
#Adding time to maturity column
expiry = pd.Timestamp("2026-01-30")
df_27["T"] = (expiry - pd.to_datetime("2026-01-27")).days / 252
df_28["T"] = (expiry - pd.to_datetime("2026-01-28")).days / 252
df_29["T"] = (expiry - pd.to_datetime("2026-01-29")).days / 252

# Adding spot to the DataFrame

meta = yf.Ticker("META")

data = meta.history(start="2026-01-01", end="2026-02-01")

spot_27 = data.loc["2026-01-27"]["Close"]
spot_28 = data.loc["2026-01-28"]["Close"]
spot_29 = data.loc["2026-01-29"]["Close"]

df_27["Spot"] = spot_27
df_28["Spot"] = spot_28
df_29["Spot"] = spot_29

print(df_27.columns)

#Adding moneyness 
for df in [df_27, df_28, df_29]:
    df["moneyness"] = df["Spot"] / df["Strike"]
    df["log_moneyness"] = np.log(df["Spot"] / df["Strike"])



Index(['Strike', 'Ticker', 'Bid', 'Ask', 'Last', 'IVM', 'Vol', 'Unnamed: 7',
       'Strike.1', 'Ticker.1', 'Bid.1', 'Ask.1', 'Last.1', 'IVM.1', 'Vol.1',
       'T', 'Spot'],
      dtype='str')


In [ ]:
#defining functions to calculate d1 and d2 for the Black-Scholes model

def _d1_d2(S, K, T, r, sigma):
    d1 = (log(S/K) + (r + (sigma**2)/2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return d1, d2

def BS_model(S, K, T, r, sigma, option_type = 'call'):
    if option_type == 'call':
        d1, d2 = _d1_d2(S, K, T, r, sigma)
        option_price = S*norm.cdf(d1) - K*exp(-r*T)*norm.cdf(d2)
    elif option_type == 'put':
        d1, d2 = _d1_d2(S, K, T, r, sigma)
        option_price = K*exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)
    else:
        raise ValueError("Invalid option type. Use 'call' or 'put'.")
    return option_price

def bs_theta(S, K, r, sigma, T, option_type='call'):
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    term1 = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))

    if option_type == 'call':
        theta = term1 - r * K * np.exp(-r * T) * norm.cdf(d2)
    else:
        theta = term1 + r * K * np.exp(-r * T) * norm.cdf(-d2)

    return theta

def theta_surface_model(option_type='call'):
    theta_values = []

    for index, row in df_27.iterrows():
        S = row['Spot']
        K = row['Strike']
        T = row['T']
        r = row['Risk Free Rate']
        sigma = row['Volatility']

        theta = bs_theta(S, K, r, sigma, T, option_type)

        theta_values.append(theta / 365)

    return theta_values